In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#DATA LOADING
import pandas as pd

# File paths
train_path = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
test_path = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"

# Load the datasets
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Display first 5 rows of training data
print(train_df.head())

In [ ]:
#boxplot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Basic structure of dataset
print("=== Data Info ===")
print(train_df.info())

print("\n=== Statistical Summary ===")
print(train_df.describe())

# 1) Histogram of SalePrice (raw and log-transformed)
plt.figure(figsize=(14,6))

plt.subplot(1,2,1)
sns.histplot(train_df["SalePrice"], kde=True, bins=40, color="skyblue")
plt.title("Histogram of SalePrice (Raw)", fontsize=14)
plt.xlabel("Sale Price", fontsize=12)
plt.ylabel("Frequency", fontsize=12)

plt.subplot(1,2,2)
sns.histplot(np.log1p(train_df["SalePrice"]), kde=True, bins=40, color="orange")
plt.title("Histogram of SalePrice (Log-Transformed)", fontsize=14)
plt.xlabel("Log(Sale Price)", fontsize=12)
plt.ylabel("Frequency", fontsize=12)

plt.tight_layout()
plt.show()

# 2) Heatmap of correlations (top 10 numeric features correlated with SalePrice)
corr_matrix = train_df.corr(numeric_only=True)
top_corr_features = corr_matrix["SalePrice"].abs().sort_values(ascending=False).head(11).index
plt.figure(figsize=(10,8))
sns.heatmap(train_df[top_corr_features].corr(), annot=True, cmap="coolwarm", fmt=".2f", cbar=True)
plt.title("Correlation Heatmap: Top 10 Numeric Features Related to SalePrice", fontsize=14)
plt.show()

# 3) Boxplot: Neighborhood vs SalePrice
plt.figure(figsize=(14,8))
sns.boxplot(x="Neighborhood", y="SalePrice", data=train_df, palette="Set3")
plt.xticks(rotation=45, ha="right")
plt.title("SalePrice Distribution by Neighborhood", fontsize=14)
plt.xlabel("Neighborhood", fontsize=12)
plt.ylabel("Sale Price", fontsize=12)
plt.show()


In [ ]:
#split train_df
from sklearn.model_selection import train_test_split

# Define features and target
X = train_df.drop("SalePrice", axis=1)
y = train_df["SalePrice"]

# Perform 80/20 train-test split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set shape:", X_train.shape)
print("Validation set shape:", X_valid.shape)
print("y_train shape:", y_train.shape)
print("y_valid shape:", y_valid.shape)


In [ ]:
#train a linear regression model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# One-hot encode categorical variables & fill missing values
X_train_enc = pd.get_dummies(X_train, drop_first=True).fillna(0)
X_valid_enc = pd.get_dummies(X_valid, drop_first=True).fillna(0)

# Align columns of validation to match training (important!)
X_valid_enc = X_valid_enc.reindex(columns=X_train_enc.columns, fill_value=0)

# Initialize and train Linear Regression model
lin_reg = LinearRegression()
lin_reg.fit(X_train_enc, y_train)

# Predictions
y_pred = lin_reg.predict(X_valid_enc)

# Evaluate model
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
r2 = r2_score(y_valid, y_pred)

print("Linear Regression Performance:")
print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

In [ ]:
#calculate R^2 score, mean squared error(MSE), and root mean squared error (RMSE) for the model
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Separate categorical and numeric features
categorical_cols = X_train.select_dtypes(include=["object"]).columns
numeric_cols = X_train.select_dtypes(exclude=["object"]).columns

# Preprocessing for numeric data
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Bundle preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# Define pipeline with Linear Regression
pipeline = Pipeline(steps=[("preprocessor", preprocessor),
                           ("model", LinearRegression())])

# Fit model
pipeline.fit(X_train, y_train)

# Predictions on validation set
y_pred = pipeline.predict(X_valid)

# Evaluation metrics
r2 = r2_score(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)

print(f"R² Score: {r2:.4f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

In [ ]:
#Suggest & implement new features. Retrain the model with them
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error

# ========================
# Feature Engineering
# ========================
train_fe = train_df.copy()
test_fe = test_df.copy()

for df in [train_fe, test_fe]:
    # Total square footage
    df["TotalSF"] = df["1stFlrSF"] + df["2ndFlrSF"] + df["TotalBsmtSF"]
    
    # Total porch area
    df["TotalPorchSF"] = (df["OpenPorchSF"] + df["EnclosedPorch"] +
                          df["3SsnPorch"] + df["ScreenPorch"])
    
    # Ratios (avoid division by zero with replace)
    df["GrLivArea_perRoom"] = np.where(
    df["TotRmsAbvGrd"] > 0, df["GrLivArea"] / df["TotRmsAbvGrd"], np.nan
     )

    df["LotArea_perRoom"] = np.where(
    df["TotRmsAbvGrd"] > 0, df["LotArea"] / df["TotRmsAbvGrd"], np.nan
    )

    df["LotArea_perRoom"] = df["LotArea"] / df["TotRmsAbvGrd"].replace(0, np.nan)
    
    # Binary flags
    df["HasBasement"] = (df["TotalBsmtSF"] > 0).astype(int)
    df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
    df["Has2ndFlr"] = (df["2ndFlrSF"] > 0).astype(int)

# Replace inf/-inf with NaN (so SimpleImputer can handle them)
train_fe = train_fe.replace([np.inf, -np.inf], np.nan)
test_fe = test_fe.replace([np.inf, -np.inf], np.nan)

# ========================
# Train/Test Split
# ========================
X = train_fe.drop("SalePrice", axis=1)
y = train_fe["SalePrice"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

categorical_cols = X_train.select_dtypes(include=["object"]).columns
numeric_cols = X_train.select_dtypes(exclude=["object"]).columns

# ========================
# Preprocessing
# ========================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# ========================
# Ridge Model
# ========================
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10, solver="lsqr"))
])

# Train
pipeline.fit(X_train, y_train)

# Predict & Evaluate
y_pred = pipeline.predict(X_valid)

r2 = r2_score(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)

print("=== Ridge Regression with New Features (Cleaned) ===")
print(f"R² Score: {r2:.4f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")


In [ ]:
#RandomForest,GardientBoosting,XGBoost
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb

# ========================
# Feature Engineering
# ========================
train_fe = train_df.copy()
test_fe = test_df.copy()

for df in [train_fe, test_fe]:
    # Total square footage
    df["TotalSF"] = df["1stFlrSF"] + df["2ndFlrSF"] + df["TotalBsmtSF"]

    # Total porch area
    df["TotalPorchSF"] = (df["OpenPorchSF"] + df["EnclosedPorch"] +
                          df["3SsnPorch"] + df["ScreenPorch"])

    # Ratios (safe with np.where)
    df["GrLivArea_perRoom"] = np.where(
        df["TotRmsAbvGrd"] > 0, df["GrLivArea"] / df["TotRmsAbvGrd"], np.nan
    )
    df["LotArea_perRoom"] = np.where(
        df["TotRmsAbvGrd"] > 0, df["LotArea"] / df["TotRmsAbvGrd"], np.nan
    )

    # Binary flags
    df["HasBasement"] = (df["TotalBsmtSF"] > 0).astype(int)
    df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
    df["Has2ndFlr"] = (df["2ndFlrSF"] > 0).astype(int)

# Replace infinities
train_fe = train_fe.replace([np.inf, -np.inf], np.nan)
test_fe = test_fe.replace([np.inf, -np.inf], np.nan)

# ========================
# Train/Test Split
# ========================
X = train_fe.drop("SalePrice", axis=1)
y = train_fe["SalePrice"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

categorical_cols = X_train.select_dtypes(include=["object"]).columns
numeric_cols = X_train.select_dtypes(exclude=["object"]).columns

# ========================
# Preprocessing
# ========================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# ========================
# Models to Compare
# ========================
models = {
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        objective="reg:squarederror"
    )
}

results = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Train
    pipeline.fit(X_train, y_train)

    # Predict & evaluate
    y_pred = pipeline.predict(X_valid)
    rmse = mean_squared_error(y_valid, y_pred, squared=False)
    results[name] = rmse

# ========================
# Print Results
# ========================
print("=== Model Comparison (Validation RMSE) ===")
for name, rmse in results.items():
    print(f"{name}: {rmse:.2f}")


In [ ]:
#submission
# Full script: train XGBoost on full training data and create submission.csv
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# ------------- Feature engineering (safe division to avoid warnings) -------------
train_fe = train_df.copy()
test_fe = test_df.copy()

for df in [train_fe, test_fe]:
    # Basic totals
    df["TotalSF"] = df["1stFlrSF"] + df["2ndFlrSF"] + df["TotalBsmtSF"]
    df["TotalPorchSF"] = df["OpenPorchSF"] + df["EnclosedPorch"] + df["3SsnPorch"] + df["ScreenPorch"]

    # Safe per-room ratios (use np.divide with where)
    df["GrLivArea_perRoom"] = np.divide(
        df["GrLivArea"].astype(float),
        df["TotRmsAbvGrd"].astype(float),
        out=np.full_like(df["GrLivArea"].astype(float), np.nan, dtype=float),
        where=(df["TotRmsAbvGrd"].astype(float) > 0)
    )
    df["LotArea_perRoom"] = np.divide(
        df["LotArea"].astype(float),
        df["TotRmsAbvGrd"].astype(float),
        out=np.full_like(df["LotArea"].astype(float), np.nan, dtype=float),
        where=(df["TotRmsAbvGrd"].astype(float) > 0)
    )

    # Binary flags
    df["HasBasement"] = (df["TotalBsmtSF"] > 0).astype(int)
    df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
    df["Has2ndFlr"] = (df["2ndFlrSF"] > 0).astype(int)

# Remove any +/-inf resulting from prior ops (converted to NaN)
train_fe.replace([np.inf, -np.inf], np.nan, inplace=True)
test_fe.replace([np.inf, -np.inf], np.nan, inplace=True)

# ------------- Prepare features and target (drop Id from features) -------------
X = train_fe.drop(["Id", "SalePrice"], axis=1)
y = train_fe["SalePrice"].copy()

# Columns to feed into ColumnTransformer
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

# ------------- Preprocessing pipelines -------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),   # fill numeric NaNs
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),  # fill categorical NaNs
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
], remainder="drop")

# ------------- XGBoost model and pipeline -------------
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror",
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

# ------------- Train on full training set -------------
print("Training XGBoost on full training data...")
pipeline.fit(X, y)

# ------------- Prepare test features and predict -------------
X_test = test_fe.drop(["Id"], axis=1)

# Ensure same columns/order exist; ColumnTransformer will handle unseen categories via handle_unknown="ignore"
preds = pipeline.predict(X_test)

# Clip negatives (shouldn't occur, but just in case) and round to nearest dollar
preds = np.maximum(0, preds)
preds = np.round(preds, 0).astype(int)

# ------------- Create submission file -------------
submission = pd.DataFrame({
    "Id": test_fe["Id"].astype(int),
    "SalePrice": preds
})

submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv (first 5 rows):")
print(submission.head())
